In [1]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm

In [2]:
alpha = 0.95
k = norm.pdf(norm.ppf(alpha)) / (1 - alpha)
print(k) 

2.0627128075074275


In [3]:
df = pd.read_csv("selected_monthly_return.csv", encoding="utf-8-sig")
tickers = df["Ticker"].tolist()
returns = df.drop(columns=["PERMNO", "Ticker", "average_monthly_return"])
mu = df["average_monthly_return"].to_numpy(dtype=float)

## Covariance Matrix

In [4]:
cov_df = pd.read_csv("covariance_matrix.csv", skiprows=1, encoding="utf-8-sig")

Sigma = cov_df.drop(columns=["PERMNO", "Ticker"]).to_numpy(dtype=float)

eigvals, eigvecs = np.linalg.eigh(Sigma)
A = np.sqrt(np.clip(eigvals, 0, None))[:, None] * eigvecs.T

In [5]:
# model
m = gp.Model("portfolio")
# weights
w = m.addMVar(len(tickers), vtype=GRB.CONTINUOUS, name="w")
# standard deviation of portfolio
s = m.addVar(vtype=GRB.CONTINUOUS, name="s")

Restricted license - for non-production use only - expires 2027-11-29


In [6]:
# constraint 1
m.addConstr(w.sum() == 1, name="full_investment")
# constraint 2
m.addConstr(w <= 0.2, name="max_weight")

<MConstr (37,) *awaiting model update*>

In [7]:
# constraint 3
benchmark = pd.read_csv("SP500_monthly_202001_202603.csv", encoding="utf-8-sig")
benchmark["Month"] = pd.to_datetime(benchmark["Month"])
b = benchmark.loc[benchmark["Month"].between("2021-01-01", "2025-12-31"), "MonthlyReturn"].mean()
m.addConstr(w @ mu >= b, name="minimum_return")

<MConstr () *awaiting model update*>

In [8]:
# constraint 4
y = m.addMVar(37, lb=-GRB.INFINITY, name="y") # helper variable for finding s 
m.addConstr(y == A @ w)
m.addGenConstrNorm(s, y, 2.0) # constraint to find s

<gurobi.GenConstr *Awaiting Model Update*>

In [9]:
m.setObjective(-1 * w @ mu + k * s, gp.GRB.MINIMIZE)

In [10]:
m.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M5 Pro
Thread count: 15 physical cores, 15 logical processors, using up to 15 threads

Optimize a model with 76 rows, 75 columns and 1517 nonzeros (Min)
Model fingerprint: 0x6c7f8d79
Model has 38 linear objective coefficients
Model has 1 simple general constraint
  1 NORM
Variable types: 75 continuous, 0 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e-06, 1e+00]
  Objective range  [2e-03, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-02, 1e+00]

Presolve removed 37 rows and 0 columns
Presolve time: 0.00s
Presolved: 40 rows, 112 columns, 1512 nonzeros
Presolved model has 37 quadratic constraint(s)
Variable types: 112 continuous, 0 integer (0 binary)
Root relaxation presolve time: 0.01s
Root relaxation presolved: 114 rows, 186 columns, 1734 nonzeros
Root relaxation presolved model has 37 second-order cone constraints
Root barrier log...

Ordering time:

In [11]:
portfolio = pd.DataFrame({"Ticker": tickers, "Optimized Weight": w.X})
print(portfolio.to_string(index=False, formatters={"Optimized Weight": "{:.6%}".format}))

Ticker Optimized Weight
  COHR        0.000001%
    GE        0.000042%
   FNV        0.000307%
    UI        0.000024%
  BABA        3.754833%
  LITE        0.000001%
  CVNA        0.000001%
   CVS        6.788201%
   NET        0.000001%
  PLTR        0.000001%
  SOFI        0.000000%
   NEM        0.000007%
   CAH       19.987988%
   GLW        0.433619%
   HWM        0.000003%
  KLAC        0.000008%
  LRCX        0.000002%
    MU        0.000004%
  INTC        0.000002%
   AMD        0.000002%
   WDC        0.000002%
     B        0.000019%
  IDXX        0.000001%
   AEM        0.000002%
  ESLT       19.999421%
  CIEN        0.000008%
   APH        0.000052%
   FIX        7.686616%
    MT        0.000001%
   CLS        0.000010%
  INSM        4.127723%
    DB        8.017730%
   STX        0.000004%
   KGC        0.000017%
   NRG        0.000026%
   WPM       14.000858%
  AVGO       15.202465%


In [12]:
expected_return = mu @ w.X

print(f"Expected portfolio return for January 2026: {expected_return:.6%}")
print(f"Portfolio monthly standard deviation: {s.X:.6%}")
print(f"Portfolio monthly 95% CVaR: {m.ObjVal:.6%}")

Expected portfolio return for January 2026: 2.997993%
Portfolio monthly standard deviation: 4.328251%
Portfolio monthly 95% CVaR: 5.929946%


In [13]:
print(f"Historical benchmark mean monthly return b: {b:.6%}") # 2021-2025
print(f"Expected portfolio monthly return: {expected_return:.6%}")
print(f"Excess return: {(expected_return - b):.6f}")
print(f"Minimum-return constraint satisfied: {'Yes' if expected_return >= b - 1e-6 else 'No'}")

Historical benchmark mean monthly return b: 1.099027%
Expected portfolio monthly return: 2.997993%
Excess return: 0.018990
Minimum-return constraint satisfied: Yes
